# Colab runner - group3-starvenus

Drives the experiments of the generated `group3-starvenus.py` on a Colab GPU. Not exported to the `.py`.

## One-time Drive layout (`My Drive/csci935/`)

```
csci935/
  Dhan-Shomadhan.zip        original, unmodified dataset zip (top-level folder Dhan-Shomadhan/)
  group3-starvenus.py       latest generated script  (upload again after every code change)
  requirements.txt          copy of the repo's requirements.txt
  pip_freeze.txt            written by this notebook the first time (use it to pin requirements.txt)
  _work/
    cache/  weights/  masks/   restored at the start of each session, saved after the first build
    results/choices.json       lr / input mode / backbone decisions (restored at the start of each session)
    runs/<EXP>/...             results synced back after every split (checkpoints excluded by default)
    runs/sweep/...             learning-rate sweep runs (never results)
    runs/env.json              GPU, Python, torch, torchvision, OpenCV versions of the last session
```

## Order of cells

1. **Setup** (run top to bottom, once per session): config and Drive mount, GPU check, copy script and
   requirements, `pip install`, unzip the dataset, environment record.
   If `pip install` changes numpy, use Runtime > Restart session and run the setup cells again.
2. **Smoke**: runs `smoke` against the unzipped original dataset and prints the GPU name. It must pass
   before you continue.
3. **Caches**: restores `cache/`, `weights/`, `masks/`, `results/` from Drive if present, then runs
   `download-weights` and `prepare` (both idempotent, quick when everything is restored) and saves anything new
   to Drive.
4. **PHASE 03 SANITY RUN** and **BATCH 1**: the first real training. The sanity run is one ResNet50 run on
   split 0 (output in `runs/scratch/`, never a result); then the learning-rate sweep, `choose-lr`, and the
   three E0 rows.
5. **EXPERIMENT BATCHES**: run only the cells you need, in matrix order (E0 first, then E1, ...). Each
   cell trains one experiment split by split and syncs `_work/runs/<EXP>/` to Drive after every split, so a
   Colab disconnect loses at most one split: run the same cell again and finished splits are skipped.
   `E1-*`, `E2-*`, `E3-*` and later rows use the best E0 input size and the best backbone, read from
   `_work/results/choices.json` (written by the gates of phase 04). Without that file they stop with a
   message that names the missing choice.
6. **Finish**: optional full sync of everything to Drive.

Handoff: download `My Drive/csci935/_work/runs/` into the local `Dhan-Shomadhan/_work/runs/` (or sync with
Google Drive desktop) and commit.

Paths in the dataset contain trailing spaces (for example `Field Background  `). The helpers below pass
paths as argument lists, never through a shell string, so no quoting is needed.

## 1. Setup (run once per session, top to bottom)

In [ ]:
# Setup: config, Drive mount, GPU check
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > select a GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0))

GROUP_NAME = "group3-starvenus"
SCRIPT = f"{GROUP_NAME}.py"
DRIVE = Path("/content/drive/MyDrive/csci935")
DRIVE_WORK = DRIVE / "_work"
DRIVE_RUNS = DRIVE_WORK / "runs"
DATA_ROOT = Path("/content/Dhan-Shomadhan")
WORK = DATA_ROOT / "_work"
SCRIPT_PATH = Path("/content") / SCRIPT
DEVICE = "cuda"
PERSISTED = ("cache", "weights", "masks", "results")  # restored from / saved to DRIVE_WORK between sessions

In [ ]:
# Setup: helpers (streaming subprocess, Drive sync)
import os


def stream(cmd, **kwargs):
    """Run a command (argument list, no shell) and print its output live. Raises on failure."""
    print("$", " ".join(str(part) for part in cmd), flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        [str(part) for part in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        **kwargs,
    )
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"command failed (exit code {code}): " + " ".join(str(part) for part in cmd[1:]))


def script(*args):
    """python group3-starvenus.py --data-root /content/Dhan-Shomadhan --device cuda <args>"""
    stream([sys.executable, SCRIPT_PATH, "--data-root", DATA_ROOT, "--device", DEVICE, *args])


def sync_tree(src, dst, keep_checkpoints=True, skip=()):
    """Copy src into dst (merge, overwrite). Skips *.pt / *.pth unless keep_checkpoints, and any `skip` names."""
    src, dst = Path(src), Path(dst)
    patterns = list(skip) + ([] if keep_checkpoints else ["*.pt", "*.pth"])
    ignore = shutil.ignore_patterns(*patterns) if patterns else None
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    print(f"synced {src} -> {dst}")


def restore_runs():
    """Bring finished runs back from Drive, so that `run` skips finished splits after a disconnect."""
    if DRIVE_RUNS.is_dir():
        sync_tree(DRIVE_RUNS, WORK / "runs")


def sync_runs(keep_checkpoints=False):
    """Copy _work/runs/ (finished splits, sweeps; not the scratch folder) to Drive."""
    sync_tree(WORK / "runs", DRIVE_RUNS, keep_checkpoints=keep_checkpoints, skip=("scratch",))


def run_batch(exp, splits=(0, 1, 2, 3, 4), keep_checkpoints=False):
    """Train and evaluate one experiment (or group such as E0) split by split, syncing to Drive after each split.

    Finished splits are skipped, so running the cell again after a disconnect resumes. Set
    keep_checkpoints=True for the experiment you need Grad-CAM or predict for (best.pt is ~100-330 MB per split).
    """
    restore_runs()
    for split in splits:
        script("run", "--exp", exp, "--splits", str(split))
        sync_runs(keep_checkpoints)


def run_sweep(exp):
    """Split-0 learning-rate sweep (lr 1e-4 and 3e-4) into _work/runs/sweep/. Never counts as a result."""
    restore_runs()
    script("run", "--exp", exp, "--sweep-lr")
    sync_runs()


def choose_lr(backbone):
    """Record the better sweep lr of a backbone in choices.json and keep that file on Drive."""
    script("choose-lr", "--backbone", backbone)
    sync_tree(WORK / "results", DRIVE_WORK / "results")
    print((WORK / "results" / "choices.json").read_text())


In [ ]:
# Setup: copy the latest script and requirements from Drive, install, unzip the dataset
for name in (SCRIPT, "requirements.txt"):
    source = DRIVE / name
    assert source.exists(), f"missing {source}: upload it to My Drive/csci935/"
    shutil.copy2(source, Path("/content") / name)
print("copied", SCRIPT, "and requirements.txt from", DRIVE)

stream([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/requirements.txt"])

freeze_path = DRIVE / "pip_freeze.txt"
if not freeze_path.exists():
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
    freeze_path.write_text(freeze.stdout)
    print("recorded", freeze_path, "- use it to pin requirements.txt")

if not DATA_ROOT.exists():
    stream(["unzip", "-q", DRIVE / "Dhan-Shomadhan.zip", "-d", "/content"])
assert DATA_ROOT.is_dir(), f"{DATA_ROOT} missing after unzip"
n_images = sum(1 for p in DATA_ROOT.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg"})
assert n_images == 1106, f"expected 1106 images in the original dataset, found {n_images}"
print("dataset OK:", n_images, "images under", DATA_ROOT)

In [ ]:
# Setup: record the environment next to the runs
import cv2
import torchvision

env = {
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "opencv": cv2.__version__,
    "cuda": torch.version.cuda,
}
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
(DRIVE_RUNS / "env.json").write_text(json.dumps(env, indent=2) + "\n")
print(json.dumps(env, indent=2))
if not platform.python_version().startswith("3.12"):
    print("WARNING: the project targets Python 3.12; Colab runs", platform.python_version())

## 2. Smoke test (must pass before continuing)

In [ ]:
# Smoke test on the original dataset structure: must print the GPU name and PASSED
print("GPU:", torch.cuda.get_device_name(0))
script("smoke")

## 3. Caches: restore from Drive, then `download-weights` and `prepare` (both idempotent)

In [ ]:
# Caches: restore caches from Drive if present, otherwise build them and save them to Drive
for name in PERSISTED:
    if (DRIVE_WORK / name).is_dir():
        sync_tree(DRIVE_WORK / name, WORK / name)

script("download-weights")  # idempotent: prints the url and stored path of every weight file
script("prepare")  # idempotent: skips cache and masks that already exist and are valid

for name in PERSISTED:
    if (WORK / name).is_dir() and not (DRIVE_WORK / name).is_dir():
        sync_tree(WORK / name, DRIVE_WORK / name)

## 4. PHASE 03 SANITY RUN (run once, before any batch)

One real run of `E0-rect` on split 0 with the full recipe (2 linear-probe + up to 30 fine-tuning epochs) at lr 3e-4.
It must finish in under ~10 minutes on a T4 and reach a validation macro-F1 clearly above chance (0.2), ideally
above 0.5. The output goes to `_work/runs/scratch/E0-rect_dev/` and is never a result (the `--lr` flag marks
it as a dev run). If the F1 stays near chance, stop: check the labels on a training-batch contact sheet, the
learning rate and the normalisation before launching any batch.

In [ ]:
# PHASE 03 SANITY RUN: E0-rect, split 0, lr 3e-4 (scratch output, not a result)
script("run", "--exp", "E0-rect", "--splits", "0", "--lr", "3e-4")
print((WORK / "runs" / "scratch" / "E0-rect_dev" / "split_0" / "history.csv").read_text())

## 4b. BATCH 1 - E0 input-size ablation (ResNet50, unified)

1. LR sweep for ResNet50 on split 0 (`E0-rect` at lr 1e-4 and 3e-4), then `choose-lr` records the better one in
   `_work/results/choices.json` (by validation mixed macro-F1). Download that file and commit it.
2. `E0-squash`, `E0-pad`, `E0-rect` on all 5 splits (below, in the E0 section).
3. Gate G0 (picking the input mode) is phase 04 work: do it from the validation results.

In [ ]:
# BATCH 1, step 1: ResNet50 learning-rate sweep on split 0 (never a result)
run_sweep("E0-rect")
choose_lr("resnet50")

# 5. EXPERIMENT BATCHES

One cell per experiment ID of the matrix in `docs/instructions/README.md`. Run only the cells you need.
Each runs all 5 splits, one at a time, and syncs `_work/runs/<ID>/` to Drive after every split.
Run a cell again after a disconnect: finished splits are skipped.

Required tier: E0, E1, E2. Tier 2: E3-A. Tier 3: S-seg, E4-B. Tier 4: E3-A-WF, X-domain, X-gradcam.

### E0 - input-size ablation (ResNet50, unified). Run all three, then pick the best on validation.

In [ ]:
run_batch("E0-squash")

In [ ]:
run_batch("E0-pad")

In [ ]:
run_batch("E0-rect")

### E1 - ResNet50 baseline and domain-wise models (best E0 input size)

In [ ]:
run_batch("E1-U")

In [ ]:
run_batch("E1-WW")

In [ ]:
run_batch("E1-FF")

In [ ]:
run_batch("E1-WF")

### E2 - architectures: ConvNeXt-T and Swin-T

In [ ]:
run_batch("E2-cnx-U")

In [ ]:
run_batch("E2-cnx-WF")

In [ ]:
run_batch("E2-swin-U")

In [ ]:
run_batch("E2-swin-WF")

### E3 - Method A: copy-paste composites (Tier 2; E3-A-WF is Tier 4)

In [ ]:
run_batch("E3-A")

In [ ]:
run_batch("E3-A-WF")

### S-seg - leaf segmenter (Tier 3)

In [ ]:
run_batch("S-seg")

### E4 - Method B: segment, crop, classify (Tier 3)

In [ ]:
run_batch("E4-B")

### X-domain - domain-specific models with the best backbone (Tier 4)

In [ ]:
run_batch("X-domain")

### X-gradcam - Grad-CAM figures (Tier 4)

In [ ]:
# X-gradcam (Tier 4): needs the checkpoint of the recommended experiment, so re-run that batch with
# keep_checkpoints=True first, e.g. run_batch("E3-A", keep_checkpoints=True).
RECOMMENDED_EXP = "E3-A"  # set after the decision gate
script("gradcam", "--exp", RECOMMENDED_EXP, "--split", "0")
sync_tree(WORK / "figures", DRIVE_WORK / "figures")

## 6. Finish

In [ ]:
# Optional: sync everything that is small to Drive (results, figures, splits; no checkpoints)
sync_runs()  # finished runs and sweeps, without checkpoints and without the scratch folder
for name in ("results", "figures", "splits"):
    if (WORK / name).is_dir():
        sync_tree(WORK / name, DRIVE_WORK / name, keep_checkpoints=False)